In [ ]:
import sys
sys.path.append('.')
from pathlib import Path
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import wfdb
from sklearn.metrics import roc_auc_score, average_precision_score
import math

# data / model
PATCH_SIZE = 50  # change to 50, 100, 250, or 500 (only thing you need to change)
SEQ_LEN = 5000 // PATCH_SIZE
IN_CHANNELS = 12
PATCH_DIM = IN_CHANNELS * PATCH_SIZE
D_MODEL = 256
NUM_HEADS = 8
NUM_LAYERS = 4
MLP_RATIO = 4
DROPOUT = 0.1

class SinCosPositionalEncoding(nn.Module):
    def __init__(self, seq_len, d_model):
        super().__init__()
        assert d_model % 2 == 0, "d_model must be even for sin-cos positional encoding"

        position = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)   # (N, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32) *
            (-math.log(10000.0) / d_model)
        )  # (D/2,)

        pe = torch.zeros(1, seq_len, d_model, dtype=torch.float32)           # (1, N, D)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe, persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)].to(dtype=x.dtype, device=x.device)

# -------------------------
# Tokenizer
# -------------------------
class FixedCNNTokenizer(nn.Module):
    def __init__(self, in_channels=12, d_model=256, patch_size=50):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv1d(
            in_channels=in_channels,
            out_channels=d_model,
            kernel_size=patch_size,
            stride=patch_size,
            bias=True,
        )

    def forward(self, x):
        # x: (B, 12, 5000)
        z = self.proj(x)          # (B, D, N)
        z = z.transpose(1, 2)     # (B, N, D)
        return z

# -------------------------
# Transformer
# -------------------------
class TransformerEncoder(nn.Module):
    def __init__(self, d_model=256, num_heads=8, num_layers=4, mlp_ratio=4, dropout=0.1):
        super().__init__()
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=d_model * mlp_ratio,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

    def forward(self, x):
        return self.encoder(x)

# -------------------------
# Contiguous masking (safe)
# -------------------------
def contiguous_token_mask(batch_size, seq_len, mask_ratio, device, span_len=3, max_tries=1000):
    """
    Returns:
      mask: (B, N) bool, True = masked
    """
    num_mask = int(round(seq_len * mask_ratio))
    mask = torch.zeros(batch_size, seq_len, dtype=torch.bool, device=device)

    for b in range(batch_size):
        tries = 0
        while int(mask[b].sum().item()) < num_mask and tries < max_tries:
            tries += 1

            remaining = num_mask - int(mask[b].sum().item())
            current_span = min(span_len, remaining)

            start_max = max(1, seq_len - current_span + 1)
            start = torch.randint(0, start_max, (1,), device=device).item()
            end = start + current_span

            # skip spans that overlap already masked region
            if mask[b, start:end].any():
                continue

            mask[b, start:end] = True

        # fallback: fill any remaining slots randomly without infinite loop risk
        if int(mask[b].sum().item()) < num_mask:
            unmasked_idx = (~mask[b]).nonzero(as_tuple=False).squeeze(1)
            remaining = num_mask - int(mask[b].sum().item())
            if len(unmasked_idx) > 0:
                pick = unmasked_idx[torch.randperm(len(unmasked_idx), device=device)[:remaining]]
                mask[b, pick] = True

    return mask

class ECGMaskedSSL(nn.Module):
    def __init__(
        self,
        in_channels=12,
        seq_len=100,
        d_model=256,
        patch_size=50,
        num_heads=8,
        num_layers=4,
        mlp_ratio=4,
        dropout=0.1,
    ):
        super().__init__()
        self.in_channels = in_channels
        self.patch_size = patch_size
        self.patch_dim = in_channels * patch_size

        self.tokenizer = FixedCNNTokenizer(in_channels, d_model, patch_size)
        self.posenc = SinCosPositionalEncoding(seq_len, d_model)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.mask_token, std=0.02)

        self.encoder = TransformerEncoder(
            d_model=d_model,
            num_heads=num_heads,
            num_layers=num_layers,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        # reconstruct raw patch values: 12 * 50 = 600 outputs per token
        self.pred_head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, self.patch_dim),
        )

    def patchify(self, x):
        """
        x: (B, C, T)
        returns raw patches: (B, N, C*P)
        """
        B, C, T = x.shape
        P = self.patch_size
        assert T % P == 0
        N = T // P
        patches = x.view(B, C, N, P).permute(0, 2, 1, 3).contiguous()   # (B, N, C, P)
        patches = patches.view(B, N, C * P)                              # (B, N, C*P)
        return patches

    def forward(self, x, mask = None, mask_ratio=0.50, span_len=5):
        tokens = self.tokenizer(x)                # (B, N, D)
        B, N, D = tokens.shape

        target_patches = self.patchify(x)         # (B, N, 600)
        if mask is None:
            mask = torch.zeros(B, N, dtype=torch.bool, device=tokens.device)

        mask_token = self.mask_token.to(dtype=tokens.dtype, device=tokens.device).expand(B, N, D)
        masked_tokens = torch.where(mask.unsqueeze(-1), mask_token, tokens)

        masked_tokens = self.posenc(masked_tokens)
        encoded = self.encoder(masked_tokens)
        pred_patches = self.pred_head(encoded)    # (B, N, 600)
        pooled = encoded.mean(dim=1)

        return {
            "pred_patches": pred_patches,
            "target_patches": target_patches,
            "mask": mask,
            "encoded": encoded,
            "pooled": pooled,
        }

In [58]:
device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')

checkpoint_path = Path(f"checkpoints_fixed{PATCH_SIZE}_ssl/best.pt")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

pretrained_model = ECGMaskedSSL(
    in_channels=IN_CHANNELS,
    seq_len=SEQ_LEN,
    d_model=D_MODEL,
    patch_size=PATCH_SIZE,
    num_heads=NUM_HEADS,
    num_layers=NUM_LAYERS,
    mlp_ratio=MLP_RATIO,
    dropout=DROPOUT,
).to(device)

state_dict = checkpoint["model_state_dict"]
state_dict = {k.replace("_orig_mod.", ""): v for k, v in state_dict.items()}
pretrained_model.load_state_dict(state_dict)
pretrained_model.eval()
print("loaded checkpoint from: ", checkpoint_path)

loaded checkpoint from:  checkpoints_fixed500_ssl/best.pt


/tmp/ipykernel_2436473/197175565.py:81: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [59]:
class PTBXLClassifier(nn.Module):
  def __init__(self, pretrained_model, feature_dim = 256, num_classes=1):
    super().__init__()
    self.pretrained_model = pretrained_model
    self.classifier = nn.Linear(feature_dim, num_classes)

  def forward(self, x):
    out = self.pretrained_model(x)
    pooled = out["pooled"]
    logits = self.classifier(pooled)
    return logits
  
model = PTBXLClassifier(pretrained_model, num_classes=5).to(device)
print(model)

PTBXLClassifier(
  (pretrained_model): ECGMaskedSSL(
    (tokenizer): FixedCNNTokenizer(
      (proj): Conv1d(12, 256, kernel_size=(500,), stride=(500,))
    )
    (posenc): SinCosPositionalEncoding()
    (encoder): TransformerEncoder(
      (encoder): TransformerEncoder(
        (layers): ModuleList(
          (0-3): 4 x TransformerEncoderLayer(
            (self_attn): MultiheadAttention(
              (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
            )
            (linear1): Linear(in_features=256, out_features=1024, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (linear2): Linear(in_features=1024, out_features=256, bias=True)
            (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
            (dropout1): Dropout(p=0.1, inplace=False)
            (dropout2): Dropout(p=0.1, inplace=False)
          )
        )
 

In [60]:
ptbxl_base = Path("/data/rohit/PTB-XL")
label_df   = pd.read_csv(ptbxl_base / "ptbxl_database.csv")
scp_df     = pd.read_csv(ptbxl_base / "scp_statements.csv", index_col=0)

label_df["scp_codes"] = label_df["scp_codes"].apply(ast.literal_eval)

SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

scp_to_superclass = {}
for scp_code, row in scp_df.iterrows():
    if row["diagnostic_class"] in SUPERCLASSES:
        scp_to_superclass[scp_code] = row["diagnostic_class"]

def get_superclass_labels(scp_codes_dict):
    labels = np.zeros(len(SUPERCLASSES), dtype=np.float32)
    for scp_code in scp_codes_dict.keys():
        if scp_code in scp_to_superclass:
            superclass = scp_to_superclass[scp_code]
            idx = SUPERCLASSES.index(superclass)
            labels[idx] = 1.0
    return labels

label_df["target"] = label_df["scp_codes"].apply(get_superclass_labels)
label_df = label_df[label_df["target"].apply(lambda x: x.sum() > 0)].copy()

label_df["ecg_path"] = label_df.apply(lambda row: ptbxl_base / row["filename_hr"], axis=1)

def files_exist(row):
    base = Path(row["ecg_path"])
    return Path(str(base) + ".hea").exists() and Path(str(base) + ".dat").exists()

label_df["file_exists"] = label_df.apply(files_exist, axis=1)
label_df = label_df[label_df["file_exists"]].copy()

train_df = label_df[label_df["strat_fold"] <= 8].copy()
val_df   = label_df[label_df["strat_fold"] == 9].copy()
test_df  = label_df[label_df["strat_fold"] == 10].copy()

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")
for i, sc in enumerate(SUPERCLASSES):
    n = label_df["target"].apply(lambda x: x[i]).sum()
    print(f"  {sc}: {int(n)}")

Train: 17111
Val:   2156
Test:  2163
  NORM: 9528
  MI: 5486
  STTC: 5250
  CD: 4907
  HYP: 2655


In [61]:
class PTBXLDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        record_path = str(row["ecg_path"])

        record = wfdb.rdrecord(record_path)
        x = record.p_signal.astype(np.float32).T   # (12, 5000)

        x = np.clip(x, -5, 5)

        mean = x.mean(axis=1, keepdims=True)
        std = x.std(axis=1, keepdims=True)
        x = (x - mean) / np.clip(std, 1e-4, None)

        x = torch.from_numpy(x).float()
        y = torch.tensor(row["target"], dtype=torch.float32)   # (5,)

        return x, y

In [62]:
train_dataset = PTBXLDataset(train_df)
val_dataset = PTBXLDataset(val_df)
test_dataset = PTBXLDataset(test_df)

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                          num_workers=8, persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False,
                          num_workers=8, persistent_workers=True, prefetch_factor=2)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False,
                          num_workers=8, persistent_workers=True, prefetch_factor=2)

print("Dataloaders ready")

Dataloaders ready


In [63]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)


def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)   # (B, 5)

        optimizer.zero_grad()
        logits = model(x)
        loss   = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_probs  = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)   # (B, 5)

            logits = model(x)
            loss   = criterion(logits, y)
            probs  = torch.sigmoid(logits)

            total_loss  += loss.item()
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)

    aucs   = []
    auprcs = []
    class_aucs   = {}
    class_auprcs = {}

    for i, sc in enumerate(SUPERCLASSES):
        if all_labels[:, i].sum() > 0:
            auc   = roc_auc_score(all_labels[:, i], all_probs[:, i])
            auprc = average_precision_score(all_labels[:, i], all_probs[:, i])
            aucs.append(auc)
            auprcs.append(auprc)
            class_aucs[sc]   = auc
            class_auprcs[sc] = auprc

    macro_auc   = np.mean(aucs)
    macro_auprc = np.mean(auprcs)

    return total_loss / len(loader), macro_auc, macro_auprc, class_aucs, class_auprcs


num_epochs   = 20
best_val_auprc = 0.0

downstream_ckpt_dir = Path(f"checkpoints_fixed{PATCH_SIZE}_downstream")
downstream_ckpt_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_auc, val_auprc, val_class_aucs, val_class_auprcs = evaluate(
        model, val_loader, criterion, device
    )

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"  Train Loss     : {train_loss:.4f}")
    print(f"  Val Loss       : {val_loss:.4f}")
    print(f"  Val Macro AUC  : {val_auc:.4f}")
    print(f"  Val Macro AUPRC: {val_auprc:.4f}")
    for sc in SUPERCLASSES:
        if sc in val_class_aucs:
            print(f"    {sc}: AUC={val_class_aucs[sc]:.4f}  AUPRC={val_class_auprcs[sc]:.4f}")
    print("-" * 40)

    if val_auprc > best_val_auprc:
        best_val_auprc = val_auprc
        torch.save(model.state_dict(), downstream_ckpt_dir / "best.pt")
        print(f"  Saved new best model (val Macro AUPRC: {val_auprc:.4f})")

best_state = torch.load(downstream_ckpt_dir / "best.pt", map_location=device)
model.load_state_dict(best_state)
print(f"\nLoaded best model (val Macro AUPRC: {best_val_auprc:.4f})")

test_loss, test_auc, test_auprc, test_class_aucs, test_class_auprcs = evaluate(
    model, test_loader, criterion, device
)

print("\n========== TEST RESULTS ==========")
print(f"Test Loss       : {round(test_loss, 4)}")
print(f"Test Macro AUC  : {round(test_auc, 4)}")
print(f"Test Macro AUPRC: {round(test_auprc, 4)}")
print("\nPer-class results:")
for sc in SUPERCLASSES:
    if sc in test_class_aucs:
        print(f"  {sc}: AUC={round(test_class_aucs[sc], 4)}  AUPRC={round(test_class_auprcs[sc], 4)}")

Epoch 1/20
  Train Loss     : 0.4728
  Val Loss       : 0.4038
  Val Macro AUC  : 0.8140
  Val Macro AUPRC: 0.5789
    NORM: AUC=0.8935  AUPRC=0.8445
    MI: AUC=0.7933  AUPRC=0.5302
    STTC: AUC=0.8728  AUPRC=0.6545
    CD: AUC=0.7931  AUPRC=0.5797
    HYP: AUC=0.7175  AUPRC=0.2856
----------------------------------------
  Saved new best model (val Macro AUPRC: 0.5789)
Epoch 2/20
  Train Loss     : 0.3886
  Val Loss       : 0.3882
  Val Macro AUC  : 0.8330
  Val Macro AUPRC: 0.6102
    NORM: AUC=0.9006  AUPRC=0.8511
    MI: AUC=0.8113  AUPRC=0.5520
    STTC: AUC=0.8917  AUPRC=0.6939
    CD: AUC=0.8207  AUPRC=0.6374
    HYP: AUC=0.7405  AUPRC=0.3168
----------------------------------------
  Saved new best model (val Macro AUPRC: 0.6102)
Epoch 3/20
  Train Loss     : 0.3692
  Val Loss       : 0.3861
  Val Macro AUC  : 0.8399
  Val Macro AUPRC: 0.6278
    NORM: AUC=0.9050  AUPRC=0.8667
    MI: AUC=0.8167  AUPRC=0.5786
    STTC: AUC=0.8933  AUPRC=0.7043
    CD: AUC=0.8297  AUPRC=0.6497